# Run the Full MethylSeg Pipeline

This notebook trains and runs a complete MethylSeg pathway on packaged example inputs for both WGBS and HM450K-style data.

Because the model is fit from scratch, this walkthrough takes longer than the saved-model examples.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import os


from methylseg import MethylSegPathway, MethylDataPrep, HMMType
from methylseg.helper_classes import DATA_DIR

## Input used in this example

MethylSeg accepts tab-delimited `.tsv` or `.tsv.gz` files.

WGBS count input uses `CpG_chrm`, `CpG_beg`, `CpG_end`, `meth`, and `coverage` columns, while HM450K-style input uses beta values directly with `CpG_chrm`, `CpG_beg`, `CpG_end`, and `beta`.

The packaged example files used below live under `data/reference_files/`.

In [ ]:
REFERENCE_DIR = DATA_DIR / Path("reference_files")

## Train and run on WGBS data

The bundled reference data is loaded from `REFERENCE_DIR`, defined above.

This first workflow prepares a WGBS sample, fits a sticky HMM pathway, runs region generation, and previews the resulting region table.

In [4]:
wgbs_test_sample_info, wgbs_test_sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "WGBS_colon-primary-tumor_1_wgbs.tsv.gz",
    sample_id="WGBS_colon-primary-tumor_1",
    resolution="wgbs",
    min_coverage=10,
    remove_low_coverage_like_cpgs=True,
).prepare()

In [ ]:
meth_seg_pathway = MethylSegPathway(
    train_sample_info=wgbs_test_sample_info,
    hmm_type=HMMType.STICKY,
    hmm_params={
        "stay_prob": 0.99995,
        "emission_mismatch_prob": 0.45,
        "fit_transitions": False,
    },
    out_dir= "out" / "full_pipeline_output_wgbs"
)

In [ ]:
region_paths = meth_seg_pathway.run_pathway()

Fitting pathway...
Generating regions ...


In [ ]:
pd.read_csv(region_paths[3], sep="\t").head()

,chr1,68238,104048,PMD
0,chr1,268411,509511,PMD
1,chr1,1110605,1153357,PMD
2,chr1,2432555,2632688,PMD
3,chr1,2773024,3251489,PMD
4,chr1,3384523,3719496,PMD


## Train and run on HM450K data

This second workflow repeats the end-to-end process for array-style input using the CT HMM configuration and writes results to a separate output directory under `out/`.

In [ ]:
hm450k_test_sample_info, hm450k_test_sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "WGBS_colon-primary-tumor_1_450k.beta.gz",
    sample_id="colon-primary-tumor_1_450k",
    resolution="450k",
    remove_low_coverage_like_cpgs=True,
).prepare()

In [ ]:
meth_seg_pathway = MethylSegPathway(
    train_sample_info=hm450k_test_sample_info,
    hmm_type=HMMType.CT,
    hmm_params= {
                    "n_emissions": 4,
                    "holding_time_guess": 1_500_000,
                    "algorithm": "forward-backward",
                    "max_iter": 25,
                    "tol": 1e-2,
                },
    out_dir= "out" / "full_pipeline_output_hm450"
)

In [ ]:
region_paths = meth_seg_pathway.run_pathway()

Fitting pathway...
Generating regions ...


In [ ]:
pd.read_csv(region_paths[3], sep="\t").head()

,chr1,995946,1001669,PMD
0,chr1,1127805,1158701,PMD
1,chr1,2684250,2888586,PMD
2,chr1,3001541,3090192,PMD
3,chr1,3553112,3560739,PMD
4,chr1,4507143,4668783,PMD
